# Notebook dedicated to comparate my selected high-energy (E>5 MeV) solar candidates with Daniel's results 

In [13]:
import numpy as np
import pickle
import glob
import io

# Functions

## Euclidean Vector Magnitude

In [22]:
def magnitude(x, y, z):
    amp = x**2 + y**2 + z**2
    mag = np.sqrt(amp)
    return mag

## Flat Lists

In [10]:
def plain_mix_list(item):
    """
    Runs recursively on any structure of list:
    - Si es lista o tupla: itera sobre sus elementos (las listas vacías se ignoran solas).
    - Si es numpy array: itera sobre sus valores planos.
    - Si es escalar (np.int32, int, etc.): emite el número directamente.
    """
    if isinstance(item, (list, tuple)):
        for sub in item:
            yield from plain_mix_list(sub)
    elif isinstance(item, np.ndarray):
        for sub in item.ravel():
            yield sub.item() if hasattr(sub, 'item') else sub
    else:
        # Escalar suelto: np.int32, int, etc.
        yield item.item() if hasattr(item, 'item') else item

# Data Cuts

In [2]:
posr_cut = 5500
en_inf_cut = 5
en_sup_cut = 14
#itr_cut = 0.20

# Load Results

## My Candidates

### 2p2PPO data

In [5]:
# ------- Directory of data -------
main_fdir = '/home/joankl/data/solars/real_data/2p2PPO/candidates/resume_files/'
#main_fdir = 'E:/Data/solars/solarnu_Realdata/2p2PPO/first_candidates/ntuple/resume_files/'

# ------- Observable Dictionary -------
obs_list = ['energy_corrected', 'posr_av', 'runID', 'eventID']

obs_dict = {obs: np.array([]) for obs in obs_list}

# ------- HS and atm IDs -------
hs_eventID_prmt = []
hs_runID_prmt = []

hs_eventID_flw = []
hs_runID_flw = []

atm_eventID_flw = []
atm_runID_flw = []

# Loop over the observable list to load the candidates data in obs_dict
for obs in obs_list:
    obs_dir = main_fdir + obs + '.npy'
    obs_i = np.load(obs_dir)
    obs_dict[obs] = np.append(obs_dict[obs], obs_i)

print(f'Nº of intial events : {len(obs_dict['energy_corrected'])}')

# ===== Apply General Cuts =====

energy_condition = (obs_dict['energy_corrected'] >= en_inf_cut) & (obs_dict['energy_corrected'] <= en_sup_cut)
posr_condition = (obs_dict['posr_av'] <= posr_cut)
runID_condition = (obs_dict['runID'] > 301000)
#itr_condition = (obs_dict['itr'] >= 0.20)

mask = (energy_condition & posr_condition & runID_condition) # & itr_condition)

runID = obs_dict['runID'][mask]
eventID = obs_dict['eventID'][mask]

# ===== Now Apply Hotspot and Atmospheric Cuts =====

# Select the format of the HS and atm files to load (pkl of npz)
load_pickel = False
load_npz = True

if load_pickel:
    # Load HS and atm results
    with open(main_fdir + 'hs_dict.pkl', 'rb') as f:
        hs_dict = pickle.load(f)
    
        hs_eventID_prmt.append(hs_dict['eventID_prompt'])
        hs_runID_prmt.append(hs_dict['runID_prompt'])
        
        hs_eventID_flw.append(hs_dict['eventID_followers'])
        hs_runID_flw.append(hs_dict['runID_followers'])
    
    with open(main_fdir + 'atm_dict.pkl', 'rb') as f:
        atm_dict = pickle.load(f)
        
        atm_eventID_flw.append(atm_dict['eventID_followers'])
        atm_runID_flw.append(atm_dict['runID_followers'])
    
    # Flatten the list of lists
    hs_eventID_prmt = np.fromiter(plain_mix_list(hs_eventID_prmt), dtype=np.int64)
    hs_runID_prmt = np.fromiter(plain_mix_list(hs_runID_prmt), dtype=np.int64)
    
    hs_eventID_flw = np.fromiter(plain_mix_list(hs_eventID_flw), dtype=np.int64)
    hs_runID_flw = np.fromiter(plain_mix_list(hs_runID_flw), dtype=np.int64)
    
    atm_eventID_flw = np.fromiter(plain_mix_list(atm_eventID_flw), dtype=np.int64)
    atm_runID_flw = np.fromiter(plain_mix_list(atm_runID_flw), dtype=np.int64)

elif load_npz:
    atm_data = np.load(main_fdir + 'atm_dict.npz', mmap_mode='r')
    hs_data = np.load(main_fdir + 'hs_dict.npz', mmap_mode='r')
    
    hs_eventID_prmt = hs_data['eventID_prompt']
    hs_runID_prmt = hs_data['runID_prompt']
    
    hs_eventID_flw = hs_data['eventID_followers']
    hs_runID_flw = hs_data['runID_followers']
    
    atm_eventID_flw = atm_data['eventID_followers']
    atm_runID_flw = atm_data['runID_followers']
    
# Unify the prompt and followers IDs of the HS
hs_eventID_full = np.concatenate((hs_eventID_prmt, hs_eventID_flw))
hs_runID_full = np.concatenate((hs_runID_prmt, hs_runID_flw))

# Remove coincident eventID and runID events with tagged HS eventID and runID
# Create an unique number such that runID*offset + eventID is an unique number

offset = np.int64(10**10)

unique_id_data = (runID.astype(np.int64) * offset) + eventID.astype(np.int64)
unique_id_hs = (hs_runID_full.astype(np.int64) * offset) + hs_eventID_full.astype(np.int64)
unique_id_atm = (atm_runID_flw.astype(np.int64) * offset) + atm_eventID_flw.astype(np.int64)

is_hotspot = np.isin(unique_id_data, unique_id_hs)
is_atmospheric = np.isin(unique_id_data, unique_id_atm)

hs_cut = ~is_hotspot
atm_cut = ~is_atmospheric

# Select the data
runID_2p2PPO_data = runID[hs_cut & atm_cut]
eventID_2p2PPO_data = eventID[hs_cut & atm_cut]

print(f'Nº of events after atm and HS cuts: {len(runID_2p2PPO_data)}')

# ====== Freed Memory ======
del obs_dict, runID, eventID

Nº of intial events : 32334
Nº of events after atm and HS cuts: 173


### bisMSB data

In [11]:
# ------- Directory of data -------
#candidates_data_dir = 'E:/Data/solars/solarnu_Realdata/bisMSB/first_candidates/analysis*/ntuple/resume_files/'
candidates_data_dir = '/home/joankl/data/solars/real_data/bisMSB/first_candidates/analysis*/ntuple/resume_files/'
candidates_data_dir_list = glob.glob(candidates_data_dir)

# ------- Observable list -------
obs_list = ['energy_corrected', 'posr_av', 'runID', 'eventID']

# ------- Observable Dictionary -------
obs_dict = {obs: np.array([]) for obs in obs_list}

# ------- HS and atm IDs -------
hs_eventID_prmt = []
hs_runID_prmt = []

hs_eventID_flw = []
hs_runID_flw = []

atm_eventID_flw = []
atm_runID_flw = []

# ------ Start loading the data ------

for candidates_data_dir_i in candidates_data_dir_list:
    for obs in obs_list:
        obs_dir = candidates_data_dir_i + obs + '.npy'
        obs_i = np.load(obs_dir)
        obs_dict[obs] = np.append(obs_dict[obs], obs_i)

print(f'Nº of intial events : {len(obs_dict['energy_corrected'])}')

# ===== Apply General Cuts =====

energy_condition = (obs_dict['energy_corrected'] >= en_inf_cut) & (obs_dict['energy_corrected'] <= en_sup_cut)
posr_condition = (obs_dict['posr_av'] <= posr_cut)
#runID_condition = (obs_dict['runID'] > 301000)
#itr_condition = (obs_dict['itr'] >= 0.20)

mask = (energy_condition & posr_condition) #& runID_condition)

runID = obs_dict['runID'][mask]
eventID = obs_dict['eventID'][mask]

print(f'Nº of events after basic cuts: {len(runID)}')

# ===== Now Apply Hotspot and atm Cuts =====

# Load HS and atm results
for candidates_data_dir_i in candidates_data_dir_list:
    with open(candidates_data_dir_i + 'hs_dict.pkl', 'rb') as f:
        hs_dict = pickle.load(f)

        hs_eventID_prmt.append(hs_dict['eventID_prompt'])
        hs_runID_prmt.append(hs_dict['runID_prompt'])
        
        hs_eventID_flw.append(hs_dict['eventID_followers'])
        hs_runID_flw.append(hs_dict['runID_followers'])
    
    with open(candidates_data_dir_i + 'atm_dict.pkl', 'rb') as f:
        atm_dict = pickle.load(f)
        
        atm_eventID_flw.append(atm_dict['eventID_followers'])
        atm_runID_flw.append(atm_dict['runID_followers'])

# Flatten the list of lists
hs_eventID_prmt = np.fromiter(plain_mix_list(hs_eventID_prmt), dtype=np.int64)
hs_runID_prmt = np.fromiter(plain_mix_list(hs_runID_prmt), dtype=np.int64)

hs_eventID_flw = np.fromiter(plain_mix_list(hs_eventID_flw), dtype=np.int64)
hs_runID_flw = np.fromiter(plain_mix_list(hs_runID_flw), dtype=np.int64)

atm_eventID_flw = np.fromiter(plain_mix_list(atm_eventID_flw), dtype=np.int64)
atm_runID_flw = np.fromiter(plain_mix_list(atm_runID_flw), dtype=np.int64)

# Unify the prompt and followers IDs of the HS
hs_eventID_full = np.concatenate((hs_eventID_prmt, hs_eventID_flw))
hs_runID_full = np.concatenate((hs_runID_prmt, hs_runID_flw))

# Remove coincident eventID and runID events with tagged HS eventID and runID
# Create an unique number such that runID*offset + eventID is an unique number

offset = np.int64(10**10)

unique_id_data = (runID.astype(np.int64) * offset) + eventID.astype(np.int64)
unique_id_hs = (hs_runID_full.astype(np.int64) * offset) + hs_eventID_full.astype(np.int64)
unique_id_atm = (atm_runID_flw.astype(np.int64) * offset) + atm_eventID_flw.astype(np.int64)

is_hotspot = np.isin(unique_id_data, unique_id_hs)
is_atmospheric = np.isin(unique_id_data, unique_id_atm)

hs_cut = ~is_hotspot
atm_cut = ~is_atmospheric

# Select the data
runID_bisMSB_data = runID[hs_cut & atm_cut]
eventID_bisMSB_data = eventID[hs_cut & atm_cut]
print(f'Nº of events after atm and HS cuts: {len(runID_bisMSB_data)}')

# ====== Freed Memory ======
del obs_dict, runID, eventID

Nº of intial events : 26136
Nº of events after basic cuts: 192
Nº of events after atm and HS cuts: 192


#### Group My Candidate and create unique Code ID using runID and eventID

In [59]:
runID = np.concatenate((runID_2p2PPO_data, runID_bisMSB_data))
eventID = np.concatenate((eventID_2p2PPO_data, eventID_bisMSB_data))

# Unique code for each event
offset = np.int64(10**10)

my_candidates_ID = (runID*offset.astype(np.int64)) + eventID.astype(np.int64)

print(f'Total of my candidates = {my_candidates_ID.shape[0]} events')

Total of my candidates = 365


## Daniel Candidates

In [62]:
fdir = '/home/joankl/data/solars/real_data/daniel_candidates/data_cut_combined_nov192025_ab212_20.dat'

daniel_data = np.loadtxt(fdir, skiprows=1)

runID = daniel_data[:, 0].astype(np.int64)
eventID = daniel_data[:, 1].astype(np.int64)     
energy = daniel_data[:, 3].astype(np.float64) 
posx = daniel_data[:, 4].astype(np.float64) 
posy = daniel_data[:, 5].astype(np.float64) 
posz = daniel_data[:, 6].astype(np.float64)
posr = magnitude(posx, posy, posz)

# ====== Data Cuts ======

energy_condition = (energy >= en_inf_cut) & (energy <= en_sup_cut)
posr_condition = (posr <= posr_cut)
runID_condition = (runID > 301000)
#itr_condition = (obs_dict['itr'] >= 0.20)

mask = (energy_condition & posr_condition & runID_condition)

runID_Dan = runID[mask]
eventID_Dan = eventID[mask]

candidates_Dan_ID = (runID_Dan*offset.astype(np.int64)) + eventID_Dan.astype(np.int64)
print(f'Daniel candidates = {candidates_Dan_ID.shape[0]} events')

Daniel candidates = 112


# Are Daniel & Martina candidates contained in my candidates?

In [67]:
test_elements_in = np.isin(candidates_Dan_ID, my_candidates_ID)

print(f'A total of {len(runID_Dan[test_elements_in])} over {candidates_Dan_ID.shape[0]} events of Daniel candidates are contained on my candidates')

A total of 89 over 112 events of Daniel candidates are contained on my candidates
